# FactoryVision — YOLO11 PCB 불량 검출 모델 학습

**실행 전 확인:** 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU` 선택

순서: 패키지 설치 → 데이터셋 다운로드 → 학습 → 성능 확인 → best.pt 다운로드

In [ ]:
# 1. 패키지 설치 + GPU 확인
%pip install -q ultralytics roboflow

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 변경하세요!)")

## 2. 데이터셋 다운로드 (Roboflow)

1. [Roboflow Universe](https://universe.roboflow.com)에서 "PCB defects" 검색 → 6개 클래스 데이터셋 선택
2. `Download Dataset` 버튼 → 포맷 **YOLOv11** 선택 → `Show download code` 선택
3. 아래 셀에 복사한 코드를 붙여넣고 실행 (api_key는 자동 포함됨)

In [ ]:
# Roboflow에서 복사한 다운로드 코드를 여기 붙여넣기 (아래는 예시)
#
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("...").project("...")
# dataset = project.version(1).download("yolov11")
#
# 실행 후 dataset.location 경로에 data.yaml이 생성된다

DATA_YAML = dataset.location + "/data.yaml"
print(DATA_YAML)

In [ ]:
# 3. 학습 (T4 GPU 기준 약 1시간 내외)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # 사전학습 모델 자동 다운로드
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    project="runs",
    name="pcb",
)

In [ ]:
# 4. 성능 확인 (목표: mAP50 >= 0.9)
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# 클래스별 성능
for i, name in metrics.names.items():
    print(f"{name}: mAP50 {metrics.box.ap50[i]:.4f}")

In [ ]:
# 5. 검출 결과 샘플 확인 (검증 이미지에 추론)
import glob
from IPython.display import Image, display

sample = glob.glob(dataset.location + "/valid/images/*")[0]
result = model.predict(sample, conf=0.25)[0]
result.save("sample_result.jpg")
display(Image("sample_result.jpg"))

In [ ]:
# 6. best.pt 다운로드 → 로컬 프로젝트의 backend/yolo/weights/best.pt 에 저장ㄴ나
from google.colab import files
files.download("runs/pcb/weights/best.pt")